In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm

model_path = "finetuned_models/jy46_fine_tuned_fake_news_bert"
batch_size = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained(model_path)
model = RobertaForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [2]:
df = pd.read_csv("./politifact/merged_politifact_test.csv")

texts = df["text"].tolist()
labels = torch.tensor(df["label"].tolist())

encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
dataset = TensorDataset(encodings["input_ids"], encodings["attention_mask"], labels)
loader = DataLoader(dataset, batch_size=batch_size)


In [3]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Evaluating"):
        input_ids, attention_mask, batch_labels = [x.to(device) for x in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

Evaluating: 100%|████████████████████████████████████████████████████████████████████| 562/562 [05:23<00:00,  1.73it/s]


### Testing Finetuned Model in Politifact


In [4]:

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
specificity = tn / (tn + fp)

print(f"\nAccuracy:    {acc:.4f}")
print(f"Precision:   {prec:.4f}")
print(f"Recall:      {rec:.4f}")
print(f"F1 Score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")


Accuracy:    0.9949
Precision:   0.9903
Recall:      0.9991
F1 Score:    0.9947
Specificity: 0.9911


### Testing Finetuned Model in ReNew


In [5]:
data_path = "cleaned dataset/renew/combined/combined renew test.csv"
batch_size = 16

df = pd.read_csv(data_path)

texts = df["statement"].tolist()
labels = torch.tensor(df["label"].tolist())

encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
dataset = TensorDataset(encodings["input_ids"], encodings["attention_mask"], labels)
loader = DataLoader(dataset, batch_size=batch_size)

In [6]:
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, attention_mask, batch_labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

In [7]:
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
specificity = tn / (tn + fp)

print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"F1 Score:     {f1:.4f}")
print(f"Specificity:  {specificity:.4f}")

Accuracy:     0.8843
Precision:    0.8537
Recall:       0.9280
F1 Score:     0.8893
Specificity:  0.8403
